In [1]:
import sys
!{sys.executable} -m pip install transformers

In [3]:
import pandas as pd
from transformers import T5Tokenizer,Trainer,TrainingArguments,T5ForConditionalGeneration

In [5]:
train_data=pd.read_csv("samsum-train.csv")
val_data=pd.read_csv("samsum-validation.csv")

In [7]:
train_data.shape

(14732, 3)

In [9]:
val_data.shape

(818, 3)

In [11]:
# random sampling(we will use 4000 values for training data and 500 values for validation data)
train_data=train_data.sample(n=4000,random_state=42).reset_index(drop=True)
val_data=val_data.sample(n=500,random_state=42).reset_index(drop=True)

In [13]:
# remove all html tags and space and /n format
import re
def clean_data(text):
    text=re.sub(r"\r\n"," ",text)
    text=re.sub(r"\s+"," ",text)
    text=re.sub(r"<.*?>"," ",text)
    text=text.strip().lower()
    return text

In [15]:
train_data["dialogue"]=train_data["dialogue"].apply(clean_data)
train_data["summary"]=train_data["summary"].apply(clean_data)
val_data["dialogue"]=val_data["dialogue"].apply(clean_data)
val_data["summary"]=val_data["summary"].apply(clean_data)

In [17]:
tokenizer=T5Tokenizer.from_pretrained("t5-small")

In [19]:
def tokenization(data):
    inputs=tokenizer(data["dialogue"],padding="max_length",max_length=512,truncation=True)
    targets=tokenizer(data["summary"],padding="max_length",max_length=150,truncation=True)
    inputs["labels"]=targets["input_ids"] #token ids=>add input as label
    return inputs

In [21]:
train_dataset=train_data.apply(tokenization,axis=1).tolist()
val_dataset=val_data.apply(tokenization,axis=1).tolist()

In [23]:
import sys
!{sys.executable} -m pip install accelerate -U

In [25]:
import sys
!{sys.executable} -m pip install transformers[torch]

In [27]:
model=T5ForConditionalGeneration.from_pretrained("t5-small")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [29]:
import torch
if torch.backends.mps.is_available():
    device="mps"
elif torch.cuda.is_available():
    device="cuda"
else:
    device="cpu"
print("device",device)
model.to(device)

device cuda


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [31]:
training_args=TrainingArguments(
    output_dir="./results",
    num_train_epochs=6,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    warmup_steps=500
)

In [33]:
trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [35]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,3.631273,0.379161
2,0.396137,0.360677
3,0.373064,0.355531
4,0.362364,0.351454
5,0.354644,0.350468
6,0.351133,0.349856


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3000, training_loss=0.9114358418782552, metrics={'train_runtime': 3280.2032, 'train_samples_per_second': 7.317, 'train_steps_per_second': 0.915, 'total_flos': 3248203235328000.0, 'train_loss': 0.9114358418782552, 'epoch': 6.0})

In [37]:
model.save_pretrained("./saved_summary/model")
tokenizer.save_pretrained("./saved_summary/model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_summary/model\\tokenizer_config.json',
 './saved_summary/model\\tokenizer.json')

In [39]:
model=T5ForConditionalGeneration.from_pretrained("./saved_summary/model")
tokenizer=T5Tokenizer.from_pretrained("./saved_summary/model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [51]:
def summarize_dialouge(dialouge):
    dialouge=clean_data(dialouge)
    #tokenize
    inputs=tokenizer(
        dialouge,
        padding="max_length",
        max_length=512,
        truncation=True,
        return_tensors="pt"
    ).to(device)
    #generate summary=>token=>ids
    model.to(device)
    targets=model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=150,
            num_beams=4,
            early_stopping=True
    )
    #token ids convert to summary=>decoding
    summary=tokenizer.decode(targets[0],skip_special_tokens=True)
    return summary

In [53]:
test_dialogue = """ 
Reporter: In today's technology news, artificial intelligence continues to expand rapidly across industries, from healthcare to finance and education. Recent reports suggest that AI adoption has significantly increased over the past few years.

Reporter: Companies are investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experiences. However, this growth has also raised questions about job displacement and ethical concerns.

Expert: AI systems are becoming more capable due to advances in deep learning and access to large datasets. These models can now perform complex tasks such as language understanding, image recognition, and even code generation.

Expert: At the same time, there are valid concerns about bias in AI models, as they often reflect the data they are trained on. Ensuring fairness and transparency is becoming a key area of research.

Reporter: Governments and organizations are beginning to introduce regulations to guide the development and deployment of AI technologies. The goal is to balance innovation with accountability.

Expert: Another challenge is explainability. Many modern AI systems, especially deep neural networks, operate as “black boxes,” making it difficult to understand how decisions are made.

Reporter: Experts also highlight the importance of responsible AI development, including data privacy, security, and long-term societal impact.

Expert: Looking ahead, collaboration between researchers, policymakers, and industry leaders will be crucial to ensure that AI systems are developed and used in a safe and beneficial way.
"""
summary=summarize_dialouge(test_dialogue)
print("summary",summary)

summary ai technology continues to expand rapidly across industries, from healthcare to finance and education. ai adoption has significantly increased over the past few years. experts highlight the importance of responsible ai development, including data privacy, security, and long-term societal impact.
